# 3. Fit two tiers and tune incident persistence on validation only
References and Isolation Forest use training data. A separate chronological normal
calibration segment maps raw scores to [0, 1]. These are interpolated ranks, not
probabilities. Statistical EWMA, Isolation Forest and their maximum are compared.
The maximum changes the false-alarm rate and is not automatically preferred.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
# Resolve relative configured output paths consistently from any notebook.
import os
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [start + pd.Timedelta(days=settings["generator"]["days"] * f)
              for f in settings["splits"]]


In [ ]:
if not (RUN / "model.joblib").exists():
    develop(CONFIG_PATH)
comparison = pd.read_csv(RUN / "validation_comparison.csv")
columns = ["detector", "opening_intervals", "closing_intervals", "detected", "faults",
           "warning_opportunities", "pre_impact_recall", "unmatched_incidents",
           "duplicate_incidents", "nuisance_per_1000_entity_days", "meets_workload_budget"]
display(comparison[columns])
manifest = json.loads((RUN / "manifest.json").read_text())
print("Frozen experimental policy:", manifest["policy"])
print("Meets validation workload budget:", manifest["meets_validation_workload_budget"])

validation_scores = pd.read_parquet(RUN / "validation_scores.parquet")
display(validation_scores[["statistical", "isolation_forest", "combined"]]
        .notna().mean().rename("score_coverage"))


Selection first prefers policies within the declared nuisance budget, then
higher pre-impact recall, then lower nuisance workload. If none passes, an
experimental policy is still saved and clearly flagged as failing the budget.
Validation is development evidence, not a final accuracy claim. New settings need
a new output folder; do not overwrite a completed experiment.